# 01 Attention 机制概述

前面已经学习了 MLP 和 CNN，并且完整跑通了 CNN-MNIST。

现在开始学习 Attention，也就是注意力机制。

这一节先不写代码，也不直接进入 Transformer。

我们先解决三个问题：

```text
Attention 为什么出现？
它到底做了什么？
它和 CNN 的关注方式有什么不同？
```

## 1. 为什么现在可以学习 Attention

Attention 并不是一套完全陌生的数学体系。

它会继续使用前面学过的知识：

- Tensor 和 shape。
- 向量与矩阵乘法。
- Softmax。
- 加权求和。
- 前向传播与反向传播。

CNN 让我们学会了从输入中提取特征。

Attention 接下来要回答的是：面对已经得到的一组信息，模型应该重点参考其中哪些部分？

## 2. Attention 要解决什么问题

一条输入里通常包含很多信息，但这些信息对当前任务不一定同等重要。

例如阅读句子：

```text
小明把复习资料交给小红，因为她明天要考试。
```

当我们理解“她”指谁时，会重点联系“小红”和“考试”，不会平均地对待每一个词。

Attention 想让模型也具备类似能力：

```text
根据当前需要，为不同信息分配不同的重要程度。
```

## 3. Attention 的核心不是删除，而是加权

Attention 不是简单地把不重要的信息全部删掉。

它通常会给每一份信息分配一个权重：

```text
更相关的信息 -> 权重大一些
不太相关的信息 -> 权重小一些
```

然后使用这些权重对信息做加权求和。

所以 Attention 可以先理解成：

```text
根据当前任务，动态决定看哪里，再把看到的信息汇总起来。
```

## 4. 用一个数字例子理解加权汇总

假设现在有三份信息，它们用三个数字简单表示：

```text
信息 A：10
信息 B：20
信息 C：30
```

模型根据当前需要分配的权重是：

```text
A 的权重：0.1
B 的权重：0.7
C 的权重：0.2
```

最后得到：

$$
0.1\times10+0.7\times20+0.2\times30=21
$$

因为 B 的权重最大，所以最终结果更接近 B。

这就是 Attention 最基础的计算直觉：不同信息以不同程度参与最终结果。

## 5. 注意力分数和注意力权重不是一回事

模型通常不会直接得到 0.1、0.7、0.2 这样的权重。

它会先计算每份信息与当前需求有多相关，得到注意力分数。

例如：

```text
原始分数：[1.0, 3.0, 2.0]
```

这些分数可以是任意实数，还不能直接当作比例。

经过 Softmax 后，才会得到非负并且总和为 1 的注意力权重。

所以要区分：

```text
注意力分数：相关程度的原始得分。
注意力权重：分数经过 Softmax 后得到的参与比例。
```

## 6. Softmax 在 Attention 中做什么

前面学习多分类时，Softmax 把多个类别分数转换成概率分布。

在 Attention 中，它做的是非常相似的事情：

```text
多个相关性分数
-> Softmax
-> 多个注意力权重
```

这些权重通常满足：

$$
\alpha_i\geq0,\qquad\sum_i\alpha_i=1
$$

因此可以把它们理解成模型这一次如何分配注意力。

Softmax 没有决定谁重要；相关性分数决定相对高低，Softmax 负责把这些分数整理成可用于加权的比例。

## 7. Attention 的最小流程

现在可以把 Attention 的基本流程压缩成三步：

```text
第 1 步：计算当前需求和各份信息的相关性分数。
第 2 步：用 Softmax 把分数变成注意力权重。
第 3 步：按照权重对信息做加权求和。
```

写成一条路线就是：

```text
相关性分数 -> Softmax -> 注意力权重 -> 加权汇总
```

后面学习 Query、Key、Value，就是把“当前需求”“用来匹配的信息”和“真正被汇总的内容”说得更精确。

## 8. 为什么说 Attention 是动态的

Attention 的权重不是训练结束后永远固定的一组数字。

同一份信息面对不同问题时，重要程度可能不同。

例如同一句话中：

- 判断“谁明天考试”时，可能重点关注人物关系。
- 判断“交付了什么”时，可能重点关注“复习资料”。

所以 Attention 的重要特点是：

```text
权重会随着输入内容和当前查询目标而变化。
```

这也是“动态加权”中“动态”两个字的含义。

## 9. Attention 和 MLP 的联系

MLP 中的 Linear 层也会做加权组合。

但 Linear 层训练完成后，连接权重作为模型参数保存下来，对不同样本使用同一组参数。

Attention 中的注意力权重则由当前输入之间的关系临时计算出来。

可以先这样对比：

```text
Linear 层参数：训练学到，之后作为模型参数反复使用。
注意力权重：根据本次输入动态计算，每次可能不同。
```

注意力权重本身通常不是直接保存的模型参数；产生这些权重所用的变换参数才会通过训练被学习。

## 10. Attention 和 CNN 的关注方式有什么不同

CNN 的卷积核先从局部窗口提取特征。

同一个卷积核会在不同位置重复使用，因此擅长发现局部模式。随着网络加深，CNN 的感受野也会逐渐扩大。

Attention 的重点则是直接计算不同位置或不同信息之间的相关性。

可以先这样比较：

```text
CNN：先按局部邻域提取特征，再通过层层堆叠扩大信息范围。
Attention：根据当前内容，直接为相关位置分配不同权重。
```

这不是说 CNN 只能看局部，也不是说 Attention 永远更好。它们只是组织和汇总信息的方式不同。

## 11. 从图像角度理解 Attention

虽然 Attention 经常从文本任务开始讲，但它也可以处理图像。

可以把图像拆成多个区域，每个区域表示一份信息。

当模型判断图片里的主体时，不同区域的重要程度可能不同：

```text
主体区域 -> 权重可能较大
无关背景 -> 权重可能较小
```

但要注意：这只是一种直觉解释。真正的注意力权重由训练数据和任务目标共同学习，不是人手工指定“哪里一定重要”。

## 12. Attention 中常见的形状

后面常把一批输入表示成：

$$
B\times N\times D
$$

其中：

- $B$：batch size。
- $N$：一条输入中有多少个位置，例如词的数量或图像块的数量。
- $D$：每个位置用多少维特征表示。

Attention 会让这 $N$ 个位置互相参考，然后为每个位置生成新的特征表示。

很多情况下，输入和输出仍然保持 `B x N x D`，但每个位置的内容已经融合了其他位置的信息。

这一节先认识三个维度，不展开矩阵计算。

## 13. Attention 也不是没有代价

Attention 能方便地建立不同位置之间的联系，但它也有代价。

如果 $N$ 个位置需要两两计算关系，相关性数量会随着 $N$ 增长得很快。

此外，单独的 Attention 不会天然知道第一个词和第二个词谁在前、谁在后。

因此后面还要学习：

- 如何控制计算量。
- 为什么需要位置编码。

现在先知道：Attention 很强，但不是没有限制。

## 14. 后续学习路线

接下来会按下面的顺序继续：

```text
Attention 的基本流程
-> Query、Key、Value
-> 注意力分数如何计算
-> Self-Attention
-> 矩阵形式和 shape
-> Multi-Head Attention
-> 位置编码
-> Transformer Encoder
```

下一课会重点学习 Query、Key、Value，弄清楚它们为什么不是三份毫无关系的数据。

## 15. 本节小结

这一节先记住：

1. 一条输入中的不同信息，对当前任务不一定同等重要。
2. Attention 会先计算相关性分数，再把分数变成权重。
3. Softmax 负责把分数整理成非负且总和为 1 的权重。
4. Attention 使用这些权重对信息做加权汇总。
5. 注意力权重会随输入和当前目标变化，因此是动态的。
6. CNN 主要从局部邻域开始提取特征，Attention 直接建模不同位置之间的相关性。
7. Attention 不是删除全部低权重信息，也不是自动等于 Transformer。

## 16. 自测问题

1. 为什么一条输入中的所有信息不应该总被同等对待？
2. Attention 为什么可以理解成动态加权汇总？
3. 注意力分数与注意力权重有什么区别？
4. Softmax 在 Attention 中做什么？
5. 注意力权重为什么会随输入变化？
6. Attention 的最小计算流程是哪三步？
7. Attention 和 Linear 层的权重有什么区别？
8. Attention 和 CNN 的关注方式有什么不同？
9. `B x N x D` 中的三个维度分别表示什么？
10. 为什么不能把 Attention 简单理解成“只保留最重要的一项”？